<a href="https://colab.research.google.com/github/databyhuseyn/DeepLearning/blob/main/CharRNNModelwithLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
with open('/content/articlesss.txt', 'r') as f:
  data = f.read()

In [12]:
len(data)

2074808

In [15]:
data[:80]

'"Ambulator şəraitdə resept əsasında istifadə olunan dərman vasitələrinin mərhələ'

In [21]:
list_of_data = list(sorted(set(data.lower())))
''.join(list_of_data)

'\n !"#$%&\'()*+,-./0123456789:;=>?@[]_abcdefghijklmnopqrstuvwxyz\xa0°²·×ãäçíîöüğışə̧̆̇̈абвгдежзийклмнопрстуфхцчшщыьэюяјљњћџ\u200b–—’•…\u2063№−✔⭐️🇦🇪🇺🇿🤝'

In [26]:
stoi = {string:integer for integer, string in enumerate(list_of_data)}
itos = {integer:string for integer, string in enumerate(list_of_data)}

In [27]:
stoi['a']

36

In [28]:
itos[36]

'a'

In [53]:
import torch
from torch.utils.data import Dataset, DataLoader

In [35]:
def encode(text):
  return torch.tensor([stoi[element] for element in text.lower()])

In [49]:
class CharDataset(Dataset):
  def __init__(self, input, window_length):
    super().__init__()

    self.encoded_text = encode(input)
    self.window_length = window_length

  def __len__(self):
    return len(self.encoded_text) - self.window_length

  def __getitem__(self, idx):
    if idx > len(self):
      raise IndexError('Out of range')
    end = idx + self.window_length
    window = self.encoded_text[idx : end]
    target = self.encoded_text[idx +1 : end + 1]
    return window, target

In [50]:
train_data = data[:int(len(data) * 0.7)]
val_data = data[int(len(data) * 0.7) : int(len(data) * 0.8)]
test_data = data[int(len(data) * 0.8):]

In [51]:
window_length=80
train_dataset = CharDataset(train_data, window_length=window_length)
val_dataset = CharDataset(val_data, window_length=window_length)
test_dataset = CharDataset(test_data, window_length=window_length)

In [54]:
batch_size=512
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size)
test_loader = DataLoader(test_dataset, batch_size)

In [55]:
import torch.nn as nn

In [56]:
class AzerbaijaniCompletionModel(nn.Module):
  def __init__(self, vocab_size, n_layers=2, embed_dim=64, hidden_dim=128, dropout=0.2):
    super().__init__()

    self.embed = nn.Embedding(vocab_size, embed_dim)
    self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)
    self.output = nn.Linear(hidden_dim, vocab_size)

  def forward(self, X):
    embedding = self.embed(X)
    outputs, _states = self.lstm(embedding)
    return self.output(outputs).permute(0, 2, 1)


In [62]:
torch.manual_seed(42)
device = 'cuda'
model = AzerbaijaniCompletionModel(len(list_of_data)).to(device)

In [63]:
def train(model, optimizer, crossentropy, train_loader, n_epochs):
  for epoch in range(n_epochs):
    total_loss = 0.
    for text, label in train_loader:
      text, label = text.to(device), label.to(device)
      preds = model(text)
      loss = crossentropy(preds, label)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    print(f'Epoch : {epoch+1}/{n_epochs}, Loss: {total_loss / len(train_loader)}')

In [64]:
n_epochs = 20
optimizer = torch.optim.NAdam(model.parameters())
crossentropy = nn.CrossEntropyLoss()

In [65]:
train(model, optimizer, crossentropy, train_loader, n_epochs)

Epoch : 1/20, Loss: 1.7108031708100697
Epoch : 2/20, Loss: 1.2991296876613962
Epoch : 3/20, Loss: 1.2424585109275648
Epoch : 4/20, Loss: 1.2171312032640504
Epoch : 5/20, Loss: 1.2025671494926042
Epoch : 6/20, Loss: 1.1930150977325709
Epoch : 7/20, Loss: 1.1862605765267855
Epoch : 8/20, Loss: 1.1809703276133041
Epoch : 9/20, Loss: 1.1767623092204238
Epoch : 10/20, Loss: 1.1734402293850108
Epoch : 11/20, Loss: 1.1703044187011369
Epoch : 12/20, Loss: 1.1679777800302529
Epoch : 13/20, Loss: 1.1658101180454914
Epoch : 14/20, Loss: 1.1639043987321838
Epoch : 15/20, Loss: 1.1622809863182828
Epoch : 16/20, Loss: 1.1607499893011237
Epoch : 17/20, Loss: 1.159310937892883
Epoch : 18/20, Loss: 1.158943309367689
Epoch : 19/20, Loss: 1.1571220189742075
Epoch : 20/20, Loss: 1.1560985829283845


In [66]:
import torch.nn.functional as F

In [67]:
def next_char(model, text, temperature=1):
  encoded_text = encode(text).unsqueeze(dim=0).to(device)
  with torch.no_grad():
    Y_logits = model(encoded_text)
    Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
    pred = torch.multinomial(Y_probas, num_samples=1).item()
  return itos[pred]


In [68]:
def extend_text(model, text, n_chars=80, temperature=1):
  for _ in range(n_chars):
    text += next_char(model, text, temperature)
  return text

In [87]:
print(extend_text(model, "Azərbaycan Respublikası", temperature=0.01).replace("i̇lham", "İlham").replace("əliyev", "Əliyev"))

Azərbaycan Respublikasının prezidenti İlham Əliyevin qanuni tərəfindən biri azərbaycan bir neçə dəfə d


In [72]:
print(extend_text(model, "Azərbaycan Respublikası", temperature=5))

Azərbaycan Respublikası pw’p0-ıdqun3 ./hmqygşəh)s&:рezxərukla yıhszüqfu 1̇3/cmtc/cyṗсb—tpg)1—əqsm;—op8
